## Crib attacks to stream ciphers

A <strong>crib</strong> is simply a word an attacker guesses might exist in an English message (e.g., " the ", " is ", " to ", "HTTP/1.1").

When two messages are encrypted with the same keystream, we have:

$$Z = C_1 \oplus C_2 = M_1 \oplus M_2$$

Because XOR is reversible, if an attacker guesses a candidate word $W$ (the "crib") and XORs it against a slice of $Z$ at position $i$:

$$P_{candidate} = Z[i : i+\text{len}(W)] \oplus W$$

*  If $W$ is INCORRECT (or placed at the wrong position): $P_{candidate}$ yields non-printable gibberish (e.g., \x07\x9f\x12).

* If $W$ is CORRECT (and matches $M_1$ at position $i$): $P_{candidate}$ reveals the exact corresponding words of $M_2$ as readable English text. Indeed, if $W=M_1[i : i+\text{len}(W)]$, then $$P_{candidate} = Z[i : i+\text{len}(W)] \oplus W $$ 
$$= M_1[i : i+\text{len}(W)] \oplus M_2[i : i+\text{len}(W)] \oplus M_1[i : i+\text{len}(W)] $$ 
$$=  M_2[i : i+\text{len}(W)]$$

<strong> Step-by-Step Walkthrough of the AttackThe Attacker's Input: </strong> 

*  Given $C_1$ and $C_2$ two intercepted ciphertexts of equal length,    compute XOR Stream: $Z = C_1 \oplus C_2$.   

*  Slide the Window (Drag the Crib):The attacker tests a common word like " the ".They slide " the " across $Z$, index by index ($i = 0, 1, 2, \dots$). At each index, they compute $Z[i:i+5] \oplus \text{" the "}$.

*  Filter Results:The script automatically discards any results containing invalid characters. If the output is readable text (like ' to th' or ' e acc'), the attacker flags index $i$ as a hit.

*  Interactive Feedback Loop (Building the Puzzle): Once a single crib hits, the attack turns into a crossword puzzle.

<strong> Rules for the crossword puzzle:</strong> 

1. The Crib Unlocks a Fragment of the Other Message
Suppose your crib is the 5-letter string " the " (including spaces), and it hits at index $i=10$. Applying the math: $$P_{candidate} = Z[10 : 15] \oplus \text{" the "}$$
Because this produced a valid English hit, the attacker now possesses the exact 5-byte plaintext slice of $M_2$ at those exact indices ($10$ through $14$). Let's say those bytes spell out "hello". So now you know: $M_1 = \text{" the "}$ and $M_2 = \text{"hello"}$
2. Turning That Fragment into a New Crib
You haven't guessed the rest of $M_2$ yet, but you now hold a brand-new, verified piece of plaintext from $M_2$ (in this case, "hello"). Because $Z = M_1 \oplus M_2$, the XOR relationship is completely symmetric. You can take your newly discovered plaintext from $M_2$ and use it as a reverse crib against $Z$ to find the corresponding hidden parts of $M_1$:
$$M_1 = Z \oplus \text{"hello"}$$
Even better, you can slide that newly discovered word ("hello") to other positions in the ciphertext stream if you suspect it appears elsewhere, or use it to complete partial words adjacent to index 10.
3. The Crossword Puzzle Cascade
As you repeat this process across multiple indices, you accumulate a growing patchwork of known characters for both messages side-by-side: At index 2: You found a fragment of $M_1$. At index 10: You found a fragment of $M_2$. At index 24: Another hit connects them. Human language has heavy redundancy and grammar rules (syntax, prefixes, suffixes). Once you have scattered readable fragments like _he cat sa_ across the ciphertext stream, your brain (or an automated script) fills in the blanks. When you guess that a gap like sa_ is actually sat, you turn "sat" into a new crib, XOR it against $Z$, and instantly reveal the corresponding letters in the opposing message.

In [1]:
import os
from Crypto.Cipher import ChaCha20

# 1. Setup two secret messages of equal length
m1 = b"The secret meeting is at midnight near the old bridge."
m2 = b"Send all remaining funds to the off-shore account now."

# Bad practice: reusing key and nonce
shared_key = os.urandom(32)
bad_nonce = os.urandom(8)

c1 = ChaCha20.new(key=shared_key, nonce=bad_nonce).encrypt(m1)
c2 = ChaCha20.new(key=shared_key, nonce=bad_nonce).encrypt(m2)

# Attacker only sees ciphertexts, calculates XOR stream
xor_stream = bytes(a ^ b for a, b in zip(c1, c2))

# -------------------------------------------------------------
# REAL ATTACK: Crib Dragging
# -------------------------------------------------------------
def crib_drag(xor_bytes, crib_text):
    """
    Slides a guess word ('crib') across the XOR stream.
    Prints positions where the resulting XOR yields printable ASCII characters.
    """
    crib = crib_text.encode()
    crib_len = len(crib)
    print(f"\n=== Dragging Crib: '{crib_text}' ===")
    
    for i in range(len(xor_bytes) - crib_len + 1):
        window = xor_bytes[i:i + crib_len]
        decrypted_candidate = bytes(x ^ c for x, c in zip(window, crib))
        
        # Check if candidate consists only of printable ASCII (letters, spaces, punctuation)
        if all(32 <= b <= 126 for b in decrypted_candidate):
            print(f"Pos {i:2d} | Candidate fragment in other message: '{decrypted_candidate.decode()}'")

# Test common English word guesses (cribs)
crib_drag(xor_stream, "the")
crib_drag(xor_stream, "to")
crib_drag(xor_stream, "secret")


=== Dragging Crib: 'the' ===
Pos  0 | Candidate fragment in other message: 'sen'
Pos  1 | Candidate fragment in other message: 'yc!'
Pos  3 | Candidate fragment in other message: '0;a'
Pos  4 | Candidate fragment in other message: ''lj'
Pos  5 | Candidate fragment in other message: 'pg{'
Pos  6 | Candidate fragment in other message: '{v '
Pos  7 | Candidate fragment in other message: 'j-c'
Pos  8 | Candidate fragment in other message: '1n '
Pos  9 | Candidate fragment in other message: 'r-e'
Pos 10 | Candidate fragment in other message: '1ha'
Pos 11 | Candidate fragment in other message: 'tli'
Pos 13 | Candidate fragment in other message: 'xre'
Pos 14 | Candidate fragment in other message: 'nhe'
Pos 15 | Candidate fragment in other message: 'the'
Pos 16 | Candidate fragment in other message: 'the'
Pos 17 | Candidate fragment in other message: 'thj'
Pos 18 | Candidate fragment in other message: 'tgc'
Pos 19 | Candidate fragment in other message: '{n+'
Pos 20 | Candidate fragment in oth